In [1]:
# =============================================================================
# CELL 1: CONFIGURATION
# =============================================================================

granularities = ['q']
START_DATE = '2025-01-01'
END_DATE = None

run_every_query = True

DATE_COL_MAP = {'q': 'book_date', 'm': 'book_date', 'w': 'app_date'}

SEGMENTS = {
    'FRN-Franchise':   {'lob_filter': 'FRN', 'dealer_type_filter': 'Franchise'},
    'FRN-Independent': {'lob_filter': 'FRN', 'dealer_type_filter': 'Independent'},
    'FLD':             {'lob_filter': 'FLD', 'dealer_type_filter': None},
    'STG':             {'lob_filter': 'STG', 'dealer_type_filter': None},
}

ROLLUP_GROUPS = {
    'FRN-Franchise + FLD + STG': ['FRN-Franchise', 'FLD', 'STG'],
    'FRN-Franchise + STG':       ['FRN-Franchise', 'STG'],
}

BASELINES = {
    'FRN-Franchise':   {'ltv': 1.94, 'new_recovery_unadjusted': 0.54, 'apr': 0.25},
    'FRN-Independent': {'ltv': 1.94, 'new_recovery_unadjusted': 0.54, 'apr': 0.25},
    'FLD':             {'ltv': 1.45, 'new_recovery_unadjusted': 0.54, 'apr': 0.235},
    'STG':             {'ltv': 1.94, 'new_recovery_unadjusted': 0.60, 'apr': 0.25},
}

MODEL_PARAMS = {
    'mmi_standard_increase': 1.03,
    'expected_years_on_book': 2,
    'impound_probability': 0.15,
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
    'skip_rate': 0.75,
    'kmx_loss_scale': 0.067,
}

EXCLUDED_VINTAGES = {}

FRN_DEALER_QUERY = """
WITH dealers AS (
    SELECT dealer_number,
           COALESCE(
               cdd.dealer_type_name,
               CASE
                   WHEN cdd.doing_business_as_name ~*
                        '\\b(honda|toyota|ford|chevrolet|chevy|gmc|dodge|chrysler|jeep|ram|buick|cadillac|lincoln|volkswagen|vw|audi|bmw|mercedes[-\\s]?benz|mercedes|benz|porsche|volvo|subaru|mazda|nissan|infiniti|lexus|acura|hyundai|kia|genesis|mitsubishi|isuzu|suzuki|tesla|rivian|lucid|ferrari|lamborghini|maserati|alfa\\s?romeo|fiat|peugeot|renault|citroen|seat|skoda|land\\s?rover|jaguar|bentley|rolls[-\\s]?royce|aston\\s?martin|mclaren|bugatti|koenigsegg|pagani|lotus|mini|smart|scion|saturn|pontiac|oldsmobile|mercury|hummer|saab|lancia|dacia|opel|vauxhall|holden|daewoo|ssangyong|geely|byd|nio|xpeng|li\\s?auto|great\\s?wall|haval|chery|dongfeng|saic|foton|jac|tata|mahindra|maruti|hero|bajaj|royal\\s?enfield|harley.?davidson|indian\\s?motorcycle|triumph)\\b'
                   THEN 'Franchise'
                   ELSE 'Independent'
               END
           ) AS dealer_type
    FROM edwnpi.crm_dealer_dim cdd
    WHERE pricing_hurdle_name = 'mROA-FRN'
      AND current_version_flag = 1
)
SELECT * FROM dealers
"""

print(f'Segments: {list(SEGMENTS.keys())}')
print(f'Rollup groups: {list(ROLLUP_GROUPS.keys())}')
print(f'Baselines: {BASELINES}')

Segments: ['FRN-Franchise', 'FRN-Independent', 'FLD', 'STG']
Rollup groups: ['FRN-Franchise + FLD + STG', 'FRN-Franchise + STG']
Baselines: {'FRN-Franchise': {'ltv': 1.94, 'new_recovery_unadjusted': 0.54, 'apr': 0.25}, 'FRN-Independent': {'ltv': 1.94, 'new_recovery_unadjusted': 0.54, 'apr': 0.25}, 'FLD': {'ltv': 1.45, 'new_recovery_unadjusted': 0.54, 'apr': 0.235}, 'STG': {'ltv': 1.94, 'new_recovery_unadjusted': 0.6, 'apr': 0.25}}


In [2]:
# =============================================================================
# CELL 2: IMPORTS AND DERIVED CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import os
import openpyxl
from tqdm.notebook import tqdm

tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)

PERIOD_FREQ_MAP = {'q': 'Q', 'm': 'M', 'w': 'W-SAT'}

date_col = DATE_COL_MAP[granularities[0]]
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()

min_date_sql = f"'{START_DATE}'"

print(f'Granularities: {granularities}')
print(f'Date column: {date_col}')
for g in granularities:
    pf = PERIOD_FREQ_MAP[g]
    print(f'  {g}: {start_date.to_period(pf)} to {end_date.to_period(pf)}')
print(f'SQL min_date: {min_date_sql}')

Granularities: ['q']
Date column: book_date
  q: 2025Q1 to 2026Q3
SQL min_date: '2025-01-01'


In [3]:
# =============================================================================
# CELL 3: UTILITY FUNCTIONS
# =============================================================================

def run_sql(filename, sub_list=None, connection=None, filename_is_query=False):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(filename, pickle_name, sub_list=None, connection=None, force_refresh=False, filename_is_query=False):
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(filename, sub_list=sub_list, connection=connection, filename_is_query=filename_is_query)
        store_pickle(df, pickle_name)
        return df
    return get_pickle(pickle_name)


def weighted_average_and_sum(group, metrics):
    if isinstance(metrics, str):
        weighted_avg = (group[metrics] * group.amt_financed).sum() / group.amt_financed.sum()
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        weighted_avg = (group[metric] * group.amt_financed).sum() / group.amt_financed.sum()
        result_dict[metric] = weighted_avg
    return pd.Series(result_dict)


def assign_period(df, col_name, freq):
    dt_series = pd.to_datetime(df[col_name])
    return dt_series.dt.to_period(freq)


def format_vintage(period_series):
    freq = period_series.iloc[0].freqstr
    if freq == 'Q-DEC':
        return period_series.dt.year.astype(str) + ' Q' + period_series.dt.quarter.astype(str)
    elif freq == 'M':
        return period_series.dt.year.astype(str) + ' M' + period_series.dt.month.astype(str).str.zfill(2)
    return period_series.astype(str)


print('Utilities ready')

Utilities ready


In [4]:
# =============================================================================
# CELL 4: ULA MULTIPLIER FUNCTION (NON-KMX ONLY)
# =============================================================================

def get_ula_multiplier_nonkmx(ula_df, leave_out='None'):
    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag
    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag
    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)
    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)
    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag
    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag
    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date
    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag
    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag
    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag
    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag
    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)
    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag
    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)
    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)
    return ula_df


print('ULA multiplier function ready')

ULA multiplier function ready


In [5]:
# =============================================================================
# CELL 5: DATA FETCH (SQL + PICKLE)
# =============================================================================

os.makedirs('cache', exist_ok=True)
force = run_every_query

need_conn = force or not all(
    os.path.exists(p) for p in ('../../cache/perm_ula.pkl', '../../cache/perm_dla.pkl', '../../cache/perm_nr.pkl', '../../cache/perm_frn_dealers.pkl')
)

if need_conn:
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        ula_df_total = cached_sql(
            '../queries/vintage_level_ula_query.txt', '../../cache/perm_ula.pkl',
            sub_list=[('{min_book_date}', f"{min_date_sql} AND dru.riskdealergroup IN ('FRN', 'FLD', 'STG')")],
            connection=conn, force_refresh=force,
        )
        print('ULA ready')

        dla_df = cached_sql(
            '../queries/new_dll_query.txt', '../../cache/perm_dla.pkl',
            connection=conn, force_refresh=force,
        )
        print('DLA ready')

        new_recovery = cached_sql(
            '../queries/new_recovery_queryt.txt', '../../cache/perm_nr.pkl',
            connection=conn, force_refresh=force,
        )
        print('New recovery ready')

        frn_dealers = cached_sql(
            FRN_DEALER_QUERY, '../../cache/perm_frn_dealers.pkl',
            connection=conn, force_refresh=force, filename_is_query=True,
        )
        print(f'FRN dealer classification ready: {len(frn_dealers):,} dealers')
else:
    ula_df_total = get_pickle('../../cache/perm_ula.pkl')
    dla_df = get_pickle('../../cache/perm_dla.pkl')
    new_recovery = get_pickle('../../cache/perm_nr.pkl')
    frn_dealers = get_pickle('../../cache/perm_frn_dealers.pkl')
    print('ULA, DLA, New recovery, FRN dealers loaded from cache')

print(f'ULA records: {len(ula_df_total):,}')
print(f'DLA records: {len(dla_df):,}')
print(f'New recovery records: {len(new_recovery):,}')
print(f'FRN dealer distribution:')
print(frn_dealers['dealer_type'].value_counts())

ULA ready
DLA ready
New recovery ready
FRN dealer classification ready: 6,118 dealers
ULA records: 835,401
DLA records: 134,149
New recovery records: 1,211,057
FRN dealer distribution:
dealer_type
Franchise      5195
Independent     922
CarMax            1
Name: count, dtype: int64


In [6]:
# =============================================================================
# CELL 6: DATA PREP, FLAG CREATION, SEGMENT FILTERING
# =============================================================================

# --- Filter out Core LOB ---
ula_df_total = ula_df_total[ula_df_total.lob != 'Core']

# --- Date columns ---
ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])

# --- Period columns ---
for df in [ula_df_total, new_recovery]:
    df['quarter'] = pd.to_datetime(df[date_col]).dt.to_period('Q')
    df['month'] = pd.to_datetime(df[date_col]).dt.to_period('M')
    df['week'] = pd.to_datetime(df[date_col]).dt.to_period('W-SAT')

for df in [ula_df_total, new_recovery]:
    df['book_week'] = df['book_week'].astype(str)
    df['app_week'] = df['app_week'].astype(str)

ula_df_total[f'{date_col}_str'] = ula_df_total[date_col].astype(str)
date_col_str = f'{date_col}_str'

# --- Flag creation ---
ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select(
    [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
     ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
    ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col_str].str[:4].astype(int)
    + ula_df_total[date_col_str].str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

# --- DLA Merge ---
dla_df = dla_df.rename(columns={'valid_vintage': 'book_vintage'})
dla_explicit = dla_df[dla_df['book_vintage'] != 'current']
dla_current = dla_df[dla_df['book_vintage'] == 'current'].drop(columns='book_vintage')
last_explicit_vintage = dla_explicit['book_vintage'].max()

ula_df_total = ula_df_total.merge(dla_explicit, how='left', on=['dealer_number', 'book_vintage'])
new_and_missing = ula_df_total['pricing_scalar'].isna() & (ula_df_total['book_vintage'] > last_explicit_vintage)
fallback = ula_df_total.loc[new_and_missing, ['dealer_number']].merge(dla_current, on='dealer_number', how='left')
for col in ['dll_edition', 'loss_ratio', 'dealer_level', 'pricing_scalar']:
    ula_df_total.loc[new_and_missing, col] = fallback[col].values

ula_df_total['pricing_scalar'] = ula_df_total['pricing_scalar'].fillna(1)
ula_df_total.loc[ula_df_total.frni_flag == 1, 'pricing_scalar'] *= 1.05
ula_df_total.loc[ula_df_total.frni_flag == 0, 'pricing_scalar'] *= 0.95

# --- Driver Flag ---
warnings.filterwarnings('ignore', category=UserWarning)
ula_df_total.job_company = ula_df_total.job_company.fillna('not provided')
ula_df_total['driver_flag'] = np.where(
    ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings('default', category=UserWarning)

# --- NA Handling ---
ula_df_total = ula_df_total.dropna(subset=['sale_price', 'cd_model_score', 'pti', 'lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# --- Weekly-matching filters ---
ula_df_total['bbltv'] = ula_df_total.amt_financed / ula_df_total.bbvalue.replace(0, np.nan)
ula_df_total = ula_df_total[ula_df_total.amt_financed <= 75000]
ula_df_total = ula_df_total[ula_df_total.pti <= 0.6]
ula_df_total = ula_df_total[
    (ula_df_total.lob == 'MCY') |
    (ula_df_total.bbltv <= 10.0) |
    (ula_df_total.bbvalue.isna()) |
    (ula_df_total.bbvalue == 0)
]
ula_df_total = ula_df_total[ula_df_total.total_income <= 200000]
print(f'ULA after weekly-matching filters: {len(ula_df_total):,}')

# --- NonKMX Flags ---
ula_df_total['ent_fld_flag'] = (ula_df_total.lob == 'ENT') | (ula_df_total.lob == 'FLD')
ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500) & (ula_df_total.lob != 'MCY')
ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY')
ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.3) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total[date_col_str] >= '2022-07-01') & (ula_df_total[date_col_str] < '2025-01-01')
ula_df_total['mcy_low_mileage_flag'] = (ula_df_total.lob == 'MCY') & (ula_df_total.cd_model_score > 140) & (ula_df_total.mileage <= 20000) & (ula_df_total.vehicle_age <= 10)
ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total['seasonal_employment_flag'] == 1
ula_df_total['waiter_employment_flag'] = ula_df_total['waiter_employment_flag'] == 1
ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['pricing_change_flag'] = ula_df_total[date_col_str] >= '2024-10-01'
ula_df_total['ent_flag'] = (ula_df_total.lob == 'ENT')
ula_df_total['student_loans_cutoff_date'] = ula_df_total[date_col_str] >= '2023-05-01'
ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & ula_df_total.lob.isin({'AN', 'FLD', 'FRN', 'STG'}) & (ula_df_total.cb_flag)
ula_df_total['student_loan_flag'] = np.where(ula_df_total.student_loan_flag == 1, 1, 0)

# --- Deduplicate driver flags ---
combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()

# --- Merge FRN dealer classification ---
ula_df_total = ula_df_total.merge(frn_dealers[['dealer_number', 'dealer_type']], on='dealer_number', how='left')

# --- Filter to relevant LOBs only ---
ula_df_total = ula_df_total[ula_df_total.lob.isin(['FRN', 'FLD', 'STG'])]
print(f'ULA after LOB filter (FRN, FLD, STG): {len(ula_df_total):,}')

# --- Segment assignment ---
conditions = [
    (ula_df_total.lob == 'FRN') & (ula_df_total.dealer_type == 'Franchise'),
    (ula_df_total.lob == 'FRN') & (ula_df_total.dealer_type != 'Franchise'),
    (ula_df_total.lob == 'FLD'),
    (ula_df_total.lob == 'STG'),
]
choices = ['FRN-Franchise', 'FRN-Independent', 'FLD', 'STG']
ula_df_total['segment'] = np.select(conditions, choices, default='Other')
ula_df_total = ula_df_total[ula_df_total.segment != 'Other']

print(f'ULA after segment assignment: {len(ula_df_total):,}')
print(f'Segment distribution:')
print(ula_df_total['segment'].value_counts())
print('[PROGRESS] Data Prep Complete')

ULA after weekly-matching filters: 830,061
ULA after LOB filter (FRN, FLD, STG): 86,441
ULA after segment assignment: 86,441
Segment distribution:
segment
FRN-Franchise      30552
STG                27395
FRN-Independent    14820
FLD                13674
Name: count, dtype: int64
[PROGRESS] Data Prep Complete


In [7]:
# =============================================================================
# CELL 7: MULTI-GRANULARITY RAGU SCORING
# =============================================================================

mean_unit_loss = MODEL_PARAMS['mean_unit_loss']
unit_loss_to_model_score = MODEL_PARAMS['unit_loss_to_model_score']

PERIOD_KEY = {'q': 'quarter', 'm': 'month', 'w': 'week'}


def get_ragu_score(vintage, ula_df_sub, new_recovery_data, ms_df, baseline_config, leave_out='None'):
    """Score a single vintage for a pre-filtered segment."""
    baseline_ltv = baseline_config['ltv']
    new_baseline_recovery_unadjusted_pct = baseline_config['new_recovery_unadjusted']
    baseline_apr = baseline_config['apr']
    ltv_mult = 17
    apr_mult = 0.7

    ula_df = ula_df_sub[ula_df_sub.vintage == vintage].copy()
    if len(ula_df) == 0:
        return None

    ula_df = get_ula_multiplier_nonkmx(ula_df, leave_out=leave_out)

    ula_df = ula_df[['account_number', date_col, 'bbvalue', 'sale_price', 'amt_financed',
                     'lob_or_bucket', 'lob', 'loss_multiplier', 'apr']]

    nr = new_recovery_data[['account_number', 'new_recovery_multiplier']].drop_duplicates(
        subset='account_number', keep='first')
    mix_df = ula_df.merge(nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
                          on='account_number', how='left').drop_duplicates(
                          subset='account_number', keep='first')
    mix_df['ltv'] = mix_df.amt_financed / mix_df.bbvalue

    bb_populated_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0)]
    if len(bb_populated_df) == 0:
        return None

    full_pop_metrics = bb_populated_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['loss_multiplier', 'ltv', 'bbvalue', 'apr'],
        include_groups=False
    )

    recovery_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0) & mix_df['recovery_multiplier'].notna()]
    recovery_df = recovery_df.copy()
    recovery_df['recovery_unadjusted_multiplier'] = recovery_df['recovery_multiplier']
    recovery_metrics = recovery_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['recovery_unadjusted_multiplier'],
        include_groups=False
    )
    recovery_metrics = recovery_metrics.drop(columns='amt_financed')

    grouped_mix_df = full_pop_metrics.join(recovery_metrics)

    vintage_ms_df = ms_df[ms_df['period'] == vintage].copy()

    index_name = grouped_mix_df.index.name
    if isinstance(index_name, str) and index_name in grouped_mix_df.columns:
        grouped_mix_df = grouped_mix_df.reset_index(drop=True)
    else:
        grouped_mix_df = grouped_mix_df.reset_index()

    full_df = grouped_mix_df.merge(vintage_ms_df, on='lob')
    if len(full_df) == 0:
        return None

    full_df['est_unit_loss'] = mean_unit_loss
    full_df['unit_loss_score'] = full_df.model_score + (1 - full_df.loss_multiplier) * full_df.est_unit_loss / unit_loss_to_model_score
    full_df = full_df.set_index('lob')

    full_df['ms_original'] = full_df.model_score.copy()
    full_df['baselined_unadjusted_recovery'] = (full_df.recovery_unadjusted_multiplier / new_baseline_recovery_unadjusted_pct).copy()

    full_df['gross_loss_impact'] = full_df['unit_loss_score'] - full_df['ms_original']
    full_df['recovery_impact'] = (full_df['unit_loss_score'] * full_df['est_unit_loss']
        * full_df['recovery_unadjusted_multiplier'] * (full_df['baselined_unadjusted_recovery'] - 1))
    full_df['ltv_impact'] = ((baseline_ltv / full_df['ltv']) - 1) * ltv_mult
    full_df['apr_impact'] = (baseline_apr - full_df['apr']) / 0.01 * apr_mult
    full_df['ragu_score'] = (
        (1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score
        + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier
        * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery
        + full_df['ltv_impact']
        + full_df['apr_impact'])

    full_df['vintage'] = vintage
    return full_df


# --- Rollup metrics ---
ROLLUP_METRICS = [
    'ms_original', 'gross_loss_impact', 'recovery_impact',
    'ltv_impact', 'apr_impact', 'ragu_score', 'ltv', 'apr',
]

# --- Main multi-granularity loop ---
all_results_by_granularity = {}

for g in granularities:
    print(f"\n{'='*60}")
    print(f"Processing granularity: {g}")
    print(f"{'='*60}")

    period_freq = PERIOD_FREQ_MAP[g]
    g_start_period = start_date.to_period(period_freq)
    g_end_period = end_date.to_period(period_freq)

    # Copy and filter by period
    ula_g = ula_df_total.copy()
    nr_g = new_recovery.copy()

    for df in [ula_g, nr_g]:
        df['period'] = df[PERIOD_KEY[g]]
        mask = (df['period'] >= g_start_period) & (df['period'] <= g_end_period)
        df.drop(df[~mask].index, inplace=True)

    # Format vintage labels
    ula_g['vintage'] = format_vintage(ula_g['period'])
    nr_g['vintage'] = format_vintage(nr_g['period'])

    all_vintages = sorted(ula_g['vintage'].unique())
    print(f"  Periods: {ula_g['period'].nunique()} | Range: {ula_g['period'].min()} to {ula_g['period'].max()}")

    # Score each segment individually
    all_segment_results = []

    for seg_name, seg_config in SEGMENTS.items():
        lob_filter = seg_config['lob_filter']
        dealer_type_filter = seg_config['dealer_type_filter']

        # Pre-filter ULA to this segment
        ula_subset = ula_g[ula_g.segment == seg_name]

        if len(ula_subset) == 0:
            print(f'  {seg_name}: no ULA loans found, skipping')
            continue

        # Compute model scores from this segment's population
        ms_subset = ula_subset.groupby(['period', 'lob']).apply(
            weighted_average_and_sum, 'cd_model_score', include_groups=False
        ).reset_index()
        ms_subset = ms_subset.rename(columns={'cd_model_score': 'model_score'})
        ms_subset['period'] = format_vintage(ms_subset['period'])

        # Score each vintage
        baseline_config = BASELINES[seg_name]
        excluded = EXCLUDED_VINTAGES.get(seg_name, set())
        seg_results = []

        for vintage in all_vintages:
            if vintage in excluded:
                continue
            try:
                result = get_ragu_score(vintage, ula_subset, nr_g, ms_subset, baseline_config)
                if result is not None:
                    seg_results.append(result)
            except Exception as e:
                print(f'    Error: {seg_name} / {vintage}: {e}')

        if not seg_results:
            print(f'  {seg_name}: no scoreable vintages')
            continue

        seg_df = pd.concat(seg_results, ignore_index=False).reset_index()
        seg_df = seg_df.rename(columns={'amt_financed_x': 'amt_financed'})

        # Rollup across lob_or_bucket within segment
        rollup = seg_df.groupby('vintage').apply(
            weighted_average_and_sum, ROLLUP_METRICS, include_groups=False
        ).reset_index()
        rollup['segment'] = seg_name
        rollup = rollup.rename(columns={'amt_financed': 'amt_financed_x'})

        all_segment_results.append(rollup)
        n_vintages = rollup.vintage.nunique()
        print(f'  {seg_name}: {n_vintages} vintages scored ({len(ula_subset):,} loans)')

    all_df = pd.concat(all_segment_results, ignore_index=True)

    # Compute rollup groups
    for group_name, group_segments in ROLLUP_GROUPS.items():
        group_data = all_df[all_df.segment.isin(group_segments)].copy()
        group_data = group_data.rename(columns={'amt_financed_x': 'amt_financed'})
        rollup = group_data.groupby('vintage').apply(
            weighted_average_and_sum, ROLLUP_METRICS, include_groups=False
        ).reset_index()
        rollup['segment'] = group_name
        rollup = rollup.rename(columns={'amt_financed': 'amt_financed_x'})
        all_df = pd.concat([all_df, rollup], ignore_index=True)
        print(f'  Rollup "{group_name}": {rollup.vintage.nunique()} vintages')

    print(f'  Total: {len(all_df)} rows across {all_df.segment.nunique()} groups and {all_df.vintage.nunique()} vintages')
    all_results_by_granularity[g] = all_df

print(f'\n[PROGRESS] All granularities complete: {granularities}')


Processing granularity: q
  Periods: 7 | Range: 2025Q1 to 2026Q3
  FRN-Franchise: 7 vintages scored (30,552 loans)
  FRN-Independent: 7 vintages scored (14,820 loans)
  FLD: 7 vintages scored (13,674 loans)
  STG: 7 vintages scored (27,395 loans)
  Rollup "FRN-Franchise + FLD + STG": 7 vintages
  Rollup "FRN-Franchise + STG": 7 vintages
  Total: 42 rows across 6 groups and 7 vintages

[PROGRESS] All granularities complete: ['q']


In [8]:
# =============================================================================
# CELL 8: DISPLAY + EXCEL EXPORT
# =============================================================================

METRIC_ROWS = [
    ('Model Score',       'ms_original'),
    ('Gross Loss Impact', 'gross_loss_impact'),
    ('Recovery Impact',   'recovery_impact'),
    ('LTV Impact',        'ltv_impact'),
    ('APR Impact',        'apr_impact'),
    ('RAGU Score',        'ragu_score'),
    ('Amount Financed',   'amt_financed_x'),
    ('Weighted LTV',      'ltv'),
    ('Weighted APR',      'apr'),
]

EXCEL_OUTPUT = '../output/lob_permutations_ragu.xlsx'
EXCEL_SHEET_MAP = {'q': 'Data Tables (Q)', 'm': 'Data Tables (M)', 'w': 'Data Tables (W)'}

all_export_segments = list(SEGMENTS.keys()) + list(ROLLUP_GROUPS.keys())

for g in granularities:
    all_df = all_results_by_granularity[g]
    sheet_name = EXCEL_SHEET_MAP[g]
    sorted_vintages = sorted(all_df['vintage'].unique())

    # --- Display summary per segment ---
    pd.set_option('display.float_format', '{:.4f}'.format)
    for seg_name in all_export_segments:
        seg_data = all_df[all_df.segment == seg_name]
        if len(seg_data) == 0:
            continue
        print(f'\n=== {seg_name} ===')
        pivot = seg_data.set_index('vintage')[['ms_original', 'gross_loss_impact',
            'recovery_impact', 'ltv_impact', 'apr_impact', 'ragu_score', 'amt_financed_x']].T
        display(pivot)

    # --- Excel export ---
    if os.path.exists(EXCEL_OUTPUT):
        wb = openpyxl.load_workbook(EXCEL_OUTPUT)
        if sheet_name in wb.sheetnames:
            del wb[sheet_name]
        ws = wb.create_sheet(sheet_name)
    else:
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = sheet_name

    current_row = 1
    for seg_name in all_export_segments:
        seg_data = all_df[all_df.segment == seg_name].set_index('vintage')
        if len(seg_data) == 0:
            continue

        ws.cell(row=current_row, column=1, value=seg_name)
        for col_idx, v in enumerate(sorted_vintages, start=2):
            ws.cell(row=current_row, column=col_idx, value=v)
        current_row += 1

        for label, col_key in METRIC_ROWS:
            ws.cell(row=current_row, column=1, value=label)
            for col_idx, v in enumerate(sorted_vintages, start=2):
                if v in seg_data.index:
                    ws.cell(row=current_row, column=col_idx, value=seg_data.loc[v, col_key])
            current_row += 1
        current_row += 1

    wb.save(EXCEL_OUTPUT)
    print(f'\nSaved to {EXCEL_OUTPUT} (sheet: {sheet_name})')
    print(f'  {len(all_export_segments)} segments x {len(sorted_vintages)} periods')

print('[PROGRESS] Excel Export Complete')


=== FRN-Franchise ===


vintage,2025 Q1,2025 Q2,2025 Q3,2025 Q4,2026 Q1,2026 Q2,2026 Q3
ms_original,138.4417,139.4604,139.8448,140.8546,142.1275,142.5130,143.3501
gross_loss_impact,2.0339,3.0058,2.9402,3.7382,3.4746,3.7388,3.7156
recovery_impact,2.4909,3.3196,3.4793,4.6143,3.3711,3.5120,3.2727
ltv_impact,5.4346,6.7210,6.5600,6.4122,5.5869,5.6733,5.2764
apr_impact,-0.3892,-0.3298,-0.3829,-0.3551,-0.2333,-0.2274,-0.2449
ragu_score,148.0118,152.1771,152.4415,155.2642,154.3267,155.2098,155.3700
amt_financed_x,103959388.4000,105311684.9000,99871100.6900,81894344.3300,116569002.7000,133176633.8500,36337723.1700



=== FRN-Independent ===


vintage,2025 Q1,2025 Q2,2025 Q3,2025 Q4,2026 Q1,2026 Q2,2026 Q3
ms_original,141.7385,142.1361,142.9318,143.7795,144.6111,145.0410,145.7503
gross_loss_impact,-1.6599,-1.3397,-0.6537,0.3736,0.2890,0.1760,1.0911
recovery_impact,-0.1107,-0.3461,0.5722,2.1305,0.8359,0.6050,0.0277
ltv_impact,4.6492,5.1327,5.2833,5.2005,3.6460,3.0059,2.7463
apr_impact,-0.6304,-0.4914,-0.6536,-1.0136,-0.8373,-0.7007,-0.8637
ragu_score,143.9867,145.0915,147.4801,150.4706,148.5448,148.1271,148.7517
amt_financed_x,70159380.8600,72608250.3300,51318850.9200,34247048.0100,41026540.0800,39082127.1300,8041337.6900



=== FLD ===


...


=== FRN-Franchise + FLD + STG ===


vintage,2025 Q1,2025 Q2,2025 Q3,2025 Q4,2026 Q1,2026 Q2,2026 Q3
ms_original,137.3555,138.5679,138.9878,140.0675,141.4534,141.8637,142.2451
gross_loss_impact,1.9468,2.4769,2.6531,2.9463,3.0405,3.2939,3.2539
recovery_impact,2.2090,2.7913,3.0373,3.9946,2.8608,3.2078,2.9959
ltv_impact,4.5430,5.4510,5.1923,5.0319,4.2006,4.5316,4.2405
apr_impact,-0.2335,-0.0744,-0.1626,-0.0882,0.0767,0.2103,0.2698
ragu_score,145.8208,149.2126,149.7079,151.9522,151.6321,153.1073,153.0051
amt_financed_x,256099036.3700,260451022.1400,231765103.4500,195571993.5700,280847247.2400,304436981.1800,91099714.7700



=== FRN-Franchise + STG ===


vintage,2025 Q1,2025 Q2,2025 Q3,2025 Q4,2026 Q1,2026 Q2,2026 Q3
ms_original,137.4964,138.6187,139.2001,140.4182,141.5865,141.8728,142.2434
gross_loss_impact,1.8758,2.4856,2.5827,3.2639,3.2463,3.4707,3.3924
recovery_impact,2.1932,2.9943,3.1858,4.3345,3.3402,3.7952,3.6294
ltv_impact,5.2991,6.3663,6.2707,6.3235,5.4202,5.5657,5.2143
apr_impact,-0.1905,0.0048,-0.0831,0.0234,0.2144,0.3189,0.4204
ragu_score,146.6739,150.4696,151.1561,154.3635,153.8076,155.0233,154.8998
amt_financed_x,212674699.6300,220531274.2000,189131180.5500,152624574.8600,220209962.8200,251523227.1500,76326049.0000



Saved to lob_permutations_ragu.xlsx (sheet: Data Tables (Q))
  6 segments x 7 periods
[PROGRESS] Excel Export Complete
